# 01 - Extract Hard Negatives

Step 01 of the TamIA pipeline in Colab format. Regenerates `data/classifier_dataset_hard` using the recall-oriented DINO candidate distribution.

Run this notebook with a GPU runtime. The code executes the same root script used by the TamIA Slurm job.

In [ ]:
# === Colab bootstrap ================================================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'clean-structure',
            'https://github.com/Azcatchi17/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
else:
    print('Local run — data/ et work_dirs/ doivent déjà être peuplés.')

→ detect GPU
✓ deps already installed, skipping
✓  /content/INF8225_Projet/data already linked
✓  /content/INF8225_Projet/work_dirs already linked
✓  /content/INF8225_Projet/MedSAM/work_dir already linked
⚠  /content/drive/Shareddrives/Projet_Medsam/grounding_dino_swin-t_pretrain_obj365_goldg_20231122_132602-4ea751ce.pth missing on Drive — skipping /content/INF8225_Projet/grounding_dino_swin-t_pretrain_obj365_goldg_20231122_132602-4ea751ce.pth

Project : /content/INF8225_Projet
Drive   : /content/drive/Shareddrives/Projet_Medsam
Device  : Tesla T4
Torch   : 2.4.0+cu121

⚠  numpy was already loaded in this kernel before setup reinstalled it.
   Runtime → Restart session, then run your imports again
   (no need to rerun setup — deps are pinned on disk).
cwd: /content/INF8225_Projet


In [ ]:
#@title Verify shared assets
from pathlib import Path

required = [
    Path("data/MSD_pancreas/train.json"),
    Path("data/MSD_pancreas/val.json"),
    Path("data/MSD_pancreas/test.json"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
    Path("msd_implementation/configs/grounding_dino/pancreas_tumor.py"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}"


OK      data/MSD_pancreas/train.json
OK      data/MSD_pancreas/val.json
OK      data/MSD_pancreas/test.json
OK      work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth
OK      work_dirs/tumor_config_v3/tumor_config_v3.py


In [ ]:
#@title Run pipeline step
import subprocess
import sys
subprocess.run([sys.executable, "-u", "-m", "msd_implementation.pipelines.resnet18_recall.extract_hard_negatives"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-u', 'extract_hard_negatives.py'], returncode=0)

In [ ]:
#@title Inspect generated classifier dataset
from pathlib import Path
base = Path("outputs/msd_implementation/resnet18_recall/datasets/classifier_dataset_resnet18")
for split in ["train", "val"]:
    for cls in ["0", "1"]:
        folder = base / split / cls
        count = len(list(folder.glob("*.png"))) if folder.exists() else 0
        print(f"{split}/{cls}: {count} patches")
assert (base / "train" / "0").exists(), "classifier_dataset_hard was not created"


train/0: 7203 patches
train/1: 2523 patches
val/0: 911 patches
val/1: 306 patches
